In [ ]:
%%configure -f
{
    "jars": ["https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar", "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar"]
}

In [ ]:
%run manage_tables

In [ ]:
from sedona.spark import SedonaContext

In [ ]:
TAXI_ZONES_SHAPEFILE_FOLDER_PATH = "Files/NYC_Taxi/taxi_zones/"

In [ ]:
def create_sedona_context(spark):
    return SedonaContext.create(spark)

def read_shapefile(sedona, file_path):
    return sedona.read.format("org.apache.sedona.sql.datasources.shapefile.ShapefileDataSource").load(file_path)

def transform_geometry_to_coordinates(df):
    return df.selectExpr("*", "ST_Transform(geometry, 'EPSG:2263', 'EPSG:4326') AS new_geometry")

def load_taxi_zones(sedona, shapefile_folder_path):
    zones_df = read_shapefile(sedona, shapefile_folder_path)
    zones_df = transform_geometry_to_coordinates(zones_df)
    create_temp_view(zones_df, "zones")

def prepare_spatial_processing(spark, shapefile_folder_path):
    sedona = create_sedona_context(spark)
    load_taxi_zones(sedona, shapefile_folder_path)
    return sedona

In [ ]:
def add_point_from_coordinates(df, latitude_column, longitude_column, new_point_column):
    return df.selectExpr("*", f"ST_Point({longitude_column}, {latitude_column}) AS {new_point_column}")

def add_zone_from_coordinates(df, sedona, latitude_column, longitude_column):
    df = add_point_from_coordinates(df, latitude_column, longitude_column, "point")
    create_temp_view(df, "df")
    return sedona.sql("""
        SELECT
            df.*,
            zones.LocationID AS location_id
        FROM df
        LEFT JOIN zones
        ON ST_Contains(zones.new_geometry, df.point)
    """).drop("point")